# Semana 8 · Tool-calling e benchmarks agênticos: um mini-BFCL em português

**Pergunta orientadora:** como avaliar separadamente a capacidade de escolher uma ferramenta, construir argumentos corretos e agir em múltiplos passos?

**Pergunta do caso:** um SLM (Qwen2.5-0.5B-Instruct, o mesmo modelo da Semana 6) consegue atuar como subagente que chama ferramentas em português? Quando falha, falha na **escolha** da função, nos **argumentos** ou na **sequência de passos**?

**Como usar:** Ambiente de execução → Alterar tipo → **GPU T4** → Executar tudo. Tempo estimado: 10 a 15 min com os dois modelos. As saídas brutas e os manifests vão para o Drive; se a sessão cair, rode de novo: o que já foi gerado com o mesmo `config_hash` é reaproveitado.

| Seção | Tópico coberto |
|---|---|
| 1 | Função, ferramenta, JSON Schema e chamada estruturada |
| 2 | Casos por categoria e gabarito no formato do BFCL (+ amostra do próprio BFCL) |
| 3 | Configuração congelada e o prompt que o modelo realmente vê |
| 4 | Geração single-turn, com a saída bruta salva |
| 5 | Parsing e validação: parser estrito × tolerante, JSON Schema |
| 6 | Checker AST e taxonomia de erros |
| 7 | Métricas por categoria: seleção, argumentos, abstenção |
| 8 | Multi-turn: executor, estado, política e pass^k |
| 9 | Ponte para a Fase 2: os mesmos passos em casos do BFCL |
| 10 | Registro de execuções e relatório final |

Referências principais: BFCL (Patil et al., ICML 2025) para as categorias e o checker AST; τ-bench (Yao et al., 2024) para avaliação por estado e pass^k; When2Call (Ross et al., NAACL 2025) para "quando não chamar".

## 0. Ambiente

In [ ]:
!pip -q install jsonschema accelerate

In [3]:
import os, re, gc, glob, json, copy, math, random, hashlib, platform, datetime, unicodedata
import importlib.metadata as imd
from collections import Counter
from itertools import permutations
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch
from jsonschema import Draft202012Validator
from IPython.display import display

try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/semana8_tool_calling"
except Exception:
    OUT = "./semana8_tool_calling"
os.makedirs(OUT, exist_ok=True)

DEV = "cuda" if torch.cuda.is_available() else "cpu"
GPU = torch.cuda.get_device_name(0) if DEV == "cuda" else "sem GPU"
ENV = dict(python=platform.python_version(), gpu=GPU,
           libs={l: imd.version(l) for l in ["torch", "transformers", "jsonschema", "numpy", "pandas"]})
print("Saída em:", OUT, "| dispositivo:", DEV, "|", GPU)
print(ENV["libs"])
if DEV == "cpu":
    print("Aviso: sem GPU a geração fica lenta. Ative a T4 em Ambiente de execução > Alterar tipo.")

# Normalização de texto usada no notebook todo: minúsculas, sem acentos, só letras e dígitos.
# O BFCL remove espaços e os caracteres ,./-_*^ ; aqui removemos toda a pontuação e os acentos
# (um pouco mais tolerante que o BFCL; isso é uma escolha de método e fica registrado).
RE_NAO_ALNUM = re.compile(r"[^a-z0-9]")
print(RE_NAO_ALNUM.sub("", unicodedata.normalize("NFKD", "Goiânia - GO").encode("ascii", "ignore").decode().lower()))

# cores fixas por modelo (a cor segue o modelo em todas as figuras)
CORES = ["#2a78d6", "#eb6834", "#1baf7a"]
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": .25, "axes.axisbelow": True, "font.size": 10})

ValueError: module functions cannot set METH_CLASS or METH_STATIC

## 1. Função, ferramenta, JSON Schema e chamada estruturada

- **Função:** código que roda de verdade (no single-turn nem precisamos dela: só comparamos a chamada com o gabarito).
- **Ferramenta:** a descrição que o modelo lê: `name`, `description` e `parameters` em **JSON Schema** (`type`, `enum`, `pattern`, `minimum`, `required`).
- **Chamada estruturada:** o que o modelo escreve: `{"name": ..., "arguments": {...}}`. O modelo nunca executa nada; quem executa é o *harness*.

O catálogo abaixo tem 12 ferramentas fictícias com contexto brasileiro. Os padrões (`pattern`) e enumerações (`enum`) tornam o formato verificável.

In [ ]:
DATA = {"type": "string", "pattern": r"^\d{4}-\d{2}-\d{2}$", "description": "Data no formato AAAA-MM-DD."}
HORA = {"type": "string", "pattern": r"^\d{2}:\d{2}$", "description": "Hora no formato HH:MM (24 h)."}
CEP = {"type": "string", "pattern": r"^[0-9]{8}$", "description": "CEP com 8 dígitos, sem traço."}
MOEDA = {"type": "string", "pattern": r"^[A-Z]{3}$", "description": "Código ISO 4217 da moeda, ex.: 'BRL', 'USD', 'EUR'."}
IATA = {"type": "string", "pattern": r"^[A-Z]{3}$", "description": "Código IATA do aeroporto, ex.: 'GYN'."}
UNIDADES = ["km", "milha", "kg", "libra", "litro", "galao"]

TOOLS = {
    "consultar_clima": {
        "description": "Retorna o tempo atual em uma cidade.",
        "parameters": {"type": "object", "properties": {
            "cidade": {"type": "string", "description": "Nome da cidade, ex.: 'Goiânia'."},
            "unidade": {"type": "string", "enum": ["celsius", "fahrenheit"], "description": "Unidade da temperatura. Padrão: celsius."}},
            "required": ["cidade"]}},
    "converter_moeda": {
        "description": "Converte um valor entre duas moedas usando a cotação do dia.",
        "parameters": {"type": "object", "properties": {
            "valor": {"type": "number", "description": "Valor a converter."},
            "origem": MOEDA, "destino": MOEDA},
            "required": ["valor", "origem", "destino"]}},
    "calcular_frete": {
        "description": "Calcula o frete de uma encomenda entre dois CEPs.",
        "parameters": {"type": "object", "properties": {
            "cep_origem": CEP, "cep_destino": CEP,
            "peso_kg": {"type": "number", "exclusiveMinimum": 0, "description": "Peso em quilogramas."}},
            "required": ["cep_origem", "cep_destino", "peso_kg"]}},
    "buscar_voos": {
        "description": "Busca voos disponíveis entre dois aeroportos em uma data.",
        "parameters": {"type": "object", "properties": {
            "origem": IATA, "destino": IATA, "data": DATA,
            "passageiros": {"type": "integer", "minimum": 1, "description": "Número de passageiros. Padrão: 1."}},
            "required": ["origem", "destino", "data"]}},
    "cotacao_acao": {
        "description": "Retorna a cotação atual de uma ação negociada na B3.",
        "parameters": {"type": "object", "properties": {
            "ticker": {"type": "string", "pattern": r"^[A-Z]{4}[0-9]{1,2}$", "description": "Código da ação, ex.: 'PETR4'."}},
            "required": ["ticker"]}},
    "traduzir_texto": {
        "description": "Traduz um texto para outro idioma.",
        "parameters": {"type": "object", "properties": {
            "texto": {"type": "string", "description": "Texto a traduzir."},
            "idioma": {"type": "string", "enum": ["en", "es", "fr", "de", "it"], "description": "Código do idioma de destino."}},
            "required": ["texto", "idioma"]}},
    "calcular_juros_compostos": {
        "description": "Calcula o montante final de uma aplicação com juros compostos.",
        "parameters": {"type": "object", "properties": {
            "capital": {"type": "number", "description": "Valor inicial, em reais."},
            "taxa_mensal": {"type": "number", "description": "Taxa de juros ao mês, em porcentagem (1.5 significa 1,5% ao mês)."},
            "meses": {"type": "integer", "minimum": 1, "description": "Número de meses."}},
            "required": ["capital", "taxa_mensal", "meses"]}},
    "buscar_receitas": {
        "description": "Busca receitas que usam um ingrediente.",
        "parameters": {"type": "object", "properties": {
            "ingrediente": {"type": "string", "description": "Ingrediente principal."},
            "tempo_max_min": {"type": "integer", "minimum": 1, "description": "Tempo máximo de preparo, em minutos."}},
            "required": ["ingrediente"]}},
    "converter_unidade": {
        "description": "Converte valores entre unidades de medida de distância, massa e volume.",
        "parameters": {"type": "object", "properties": {
            "valor": {"type": "number", "description": "Valor a converter."},
            "de": {"type": "string", "enum": UNIDADES, "description": "Unidade de origem."},
            "para": {"type": "string", "enum": UNIDADES, "description": "Unidade de destino."}},
            "required": ["valor", "de", "para"]}},
    "criar_lembrete": {
        "description": "Cria um lembrete em uma data e hora.",
        "parameters": {"type": "object", "properties": {
            "texto": {"type": "string", "description": "O que deve ser lembrado."},
            "data": DATA, "hora": HORA},
            "required": ["texto", "data", "hora"]}},
    "consultar_cep": {
        "description": "Retorna o endereço correspondente a um CEP.",
        "parameters": {"type": "object", "properties": {"cep": CEP}, "required": ["cep"]}},
    "buscar_noticias": {
        "description": "Busca notícias recentes sobre um tema.",
        "parameters": {"type": "object", "properties": {
            "tema": {"type": "string", "description": "Tema da busca."},
            "quantidade": {"type": "integer", "minimum": 1, "maximum": 10, "description": "Quantas notícias retornar. Padrão: 5."}},
            "required": ["tema"]}},
}
for nome in TOOLS:
    TOOLS[nome] = {"name": nome, **TOOLS[nome]}

print(len(TOOLS), "ferramentas:", ", ".join(TOOLS))
print(json.dumps({"type": "function", "function": TOOLS["converter_moeda"]}, ensure_ascii=False, indent=2))

In [ ]:
# Uma chamada estruturada é só um JSON com nome e argumentos.
# O schema diz o que é VÁLIDO, não o que é CERTO.
pedido = "Quanto valem 1.200 euros em reais?"
validador = Draft202012Validator(TOOLS["converter_moeda"]["parameters"])
tentativas = {
    "certa": {"valor": 1200, "origem": "EUR", "destino": "BRL"},
    "válida, mas errada": {"valor": 1.2, "origem": "EUR", "destino": "BRL"},   # leu "1.200" como 1,2
    "inválida": {"valor": "1.200", "origem": "euro", "destino": "BRL"},
}
print("Pedido:", pedido)
for rotulo, args in tentativas.items():
    erros = [e.message for e in validador.iter_errors(args)]
    print(f"  {rotulo:20s} schema {'OK ' if not erros else 'NÃO'}  {erros}")

## 2. Casos por categoria e gabarito

Seis categorias, dez casos cada (60 no total). As quatro primeiras seguem o BFCL; as duas últimas testam **não chamar**:

| Categoria | Ferramentas oferecidas | Chamadas esperadas | O que testa |
|---|---|---|---|
| `simples` | 1 | 1 | preencher argumentos |
| `multiplas` | 3 | 1 | escolher a função certa entre distratores |
| `paralelas` | 1 | 2 a 3 da mesma função | repetir a chamada com valores diferentes |
| `paralelas_multiplas` | 3 | 2 a 3 de funções diferentes | escolher e repetir |
| `irrelevancia` | 2 a 3 | 0 | não forçar uma ferramenta que não serve |
| `falta_info` | 1 | 0 | não inventar o que o usuário não disse (perguntar) |

**Gabarito no formato do BFCL** (`possible_answer`): `{"funcao": {"parametro": [valores aceitos]}}`.
- `""` na lista: o parâmetro é opcional e pode ser omitido. Se o modelo passar um valor, ele precisa estar na lista (por isso o valor padrão aparece junto: `["", "celsius"]`).
- `"*"` (extensão nossa): qualquer texto não vazio, para campos de texto livre como o texto de um lembrete.
- Strings comparadas depois da normalização; números comparados como números (inteiro é aceito onde se espera `number`, mas não o contrário).
- Chamadas paralelas: a ordem não importa, e a avaliação é tudo-ou-nada.

"Hoje" é fixado no prompt de sistema (quarta-feira, 2026-11-04) para que datas relativas tenham uma resposta certa.

In [ ]:
GOIANIA = ["Goiânia", "Goiânia, GO", "Goiânia, Goiás", "Goiânia, Brasil", "Goiânia, GO, Brasil"]
RECIFE = ["Recife", "Recife, PE", "Recife, Pernambuco", "Recife, Brasil"]
BRASILIA = ["Brasília", "Brasília, DF", "Brasília, Distrito Federal", "Brasília, Brasil"]
CUIABA = ["Cuiabá", "Cuiabá, MT", "Cuiabá, Mato Grosso", "Cuiabá, Brasil"]
LISBOA = ["Lisboa", "Lisboa, Portugal"]
CEL = ["", "celsius"]

CASOS_PT = [
    # ---------- simples: 1 ferramenta, 1 chamada ----------
    ("simples_01", "Como está o tempo em Goiânia agora?", ["consultar_clima"],
     [{"consultar_clima": {"cidade": GOIANIA, "unidade": CEL}}]),
    ("simples_02", "Quanto dá 250 dólares em reais?", ["converter_moeda"],
     [{"converter_moeda": {"valor": [250], "origem": ["USD"], "destino": ["BRL"]}}]),
    ("simples_03", "Quanto custa o frete de um pacote de 2,5 kg do CEP 74605-010 para o CEP 01310-100?", ["calcular_frete"],
     [{"calcular_frete": {"cep_origem": ["74605010"], "cep_destino": ["01310100"], "peso_kg": [2.5]}}]),
    ("simples_04", "Procure voos de Goiânia (GYN) para Brasília (BSB) no dia 20/12/2026 para 2 pessoas.", ["buscar_voos"],
     [{"buscar_voos": {"origem": ["GYN"], "destino": ["BSB"], "data": ["2026-12-20"], "passageiros": [2]}}]),
    ("simples_05", "Qual a cotação da PETR4 agora?", ["cotacao_acao"],
     [{"cotacao_acao": {"ticker": ["PETR4"]}}]),
    ("simples_06", "Traduza para o inglês: O relatório fica pronto na sexta-feira.", ["traduzir_texto"],
     [{"traduzir_texto": {"texto": ["O relatório fica pronto na sexta-feira."], "idioma": ["en"]}}]),
    ("simples_07", "Quanto rende uma aplicação de R$ 5.000 a 1,2% ao mês durante 12 meses?", ["calcular_juros_compostos"],
     [{"calcular_juros_compostos": {"capital": [5000], "taxa_mensal": [1.2], "meses": [12]}}]),
    ("simples_08", "Me sugira receitas com abóbora.", ["buscar_receitas"],
     [{"buscar_receitas": {"ingrediente": ["abóbora"], "tempo_max_min": [""]}}]),
    ("simples_09", "Converta 42 km para milhas.", ["converter_unidade"],
     [{"converter_unidade": {"valor": [42], "de": ["km"], "para": ["milha"]}}]),
    ("simples_10", "Me lembre amanhã às 9h de enviar o relatório para o Lucca.", ["criar_lembrete"],
     [{"criar_lembrete": {"texto": ["*"], "data": ["2026-11-05"], "hora": ["09:00"]}}]),

    # ---------- multiplas: 3 ferramentas, 1 chamada ----------
    ("multiplas_01", "Vai chover em Recife hoje?", ["consultar_clima", "buscar_noticias", "consultar_cep"],
     [{"consultar_clima": {"cidade": RECIFE, "unidade": CEL}}]),
    ("multiplas_02", "Qual é o endereço do CEP 70040-010?", ["consultar_cep", "calcular_frete", "consultar_clima"],
     [{"consultar_cep": {"cep": ["70040010"]}}]),
    ("multiplas_03", "Quanto valem 1.200 euros em reais?", ["converter_moeda", "cotacao_acao", "calcular_juros_compostos"],
     [{"converter_moeda": {"valor": [1200], "origem": ["EUR"], "destino": ["BRL"]}}]),
    ("multiplas_04", "Me traga 3 notícias sobre o Pix.", ["buscar_noticias", "traduzir_texto", "cotacao_acao"],
     [{"buscar_noticias": {"tema": ["Pix", "o Pix"], "quantidade": [3]}}]),
    ("multiplas_05", "Procure voos de Guarulhos (GRU) para Lisboa (LIS) em 15 de janeiro de 2027.", ["buscar_voos", "converter_moeda", "consultar_clima"],
     [{"buscar_voos": {"origem": ["GRU"], "destino": ["LIS"], "data": ["2027-01-15"], "passageiros": ["", 1]}}]),
    ("multiplas_06", "Quanto são 10 libras em quilos?", ["converter_unidade", "converter_moeda", "calcular_frete"],
     [{"converter_unidade": {"valor": [10], "de": ["libra"], "para": ["kg"]}}]),
    ("multiplas_07", "Quanto está valendo a ação VALE3 hoje?", ["cotacao_acao", "buscar_noticias", "converter_moeda"],
     [{"cotacao_acao": {"ticker": ["VALE3"]}}]),
    ("multiplas_08", "Traduza 'bom dia, equipe' para o espanhol.", ["traduzir_texto", "buscar_noticias", "criar_lembrete"],
     [{"traduzir_texto": {"texto": ["bom dia, equipe"], "idioma": ["es"]}}]),
    ("multiplas_09", "Quero uma receita com frango que fique pronta em até 30 minutos.", ["buscar_receitas", "buscar_noticias", "consultar_clima"],
     [{"buscar_receitas": {"ingrediente": ["frango"], "tempo_max_min": [30]}}]),
    ("multiplas_10", "Crie um lembrete para 10/11/2026 às 14:30: reunião com a equipe de benchmarks.", ["criar_lembrete", "buscar_voos", "buscar_noticias"],
     [{"criar_lembrete": {"texto": ["*"], "data": ["2026-11-10"], "hora": ["14:30"]}}]),

    # ---------- paralelas: 1 ferramenta, várias chamadas ----------
    ("paralelas_01", "Como está o tempo em Goiânia, Brasília e Cuiabá?", ["consultar_clima"],
     [{"consultar_clima": {"cidade": GOIANIA, "unidade": CEL}}, {"consultar_clima": {"cidade": BRASILIA, "unidade": CEL}},
      {"consultar_clima": {"cidade": CUIABA, "unidade": CEL}}]),
    ("paralelas_02", "Converta 100 dólares e 100 euros para reais.", ["converter_moeda"],
     [{"converter_moeda": {"valor": [100], "origem": ["USD"], "destino": ["BRL"]}},
      {"converter_moeda": {"valor": [100], "origem": ["EUR"], "destino": ["BRL"]}}]),
    ("paralelas_03", "Quais as cotações de PETR4, VALE3 e ITUB4?", ["cotacao_acao"],
     [{"cotacao_acao": {"ticker": ["PETR4"]}}, {"cotacao_acao": {"ticker": ["VALE3"]}}, {"cotacao_acao": {"ticker": ["ITUB4"]}}]),
    ("paralelas_04", "Traduza 'obrigado' para o inglês e para o francês.", ["traduzir_texto"],
     [{"traduzir_texto": {"texto": ["obrigado"], "idioma": ["en"]}}, {"traduzir_texto": {"texto": ["obrigado"], "idioma": ["fr"]}}]),
    ("paralelas_05", "Calcule o frete de um pacote de 1 kg e de outro de 5 kg, ambos do CEP 74000000 para o CEP 20040002.", ["calcular_frete"],
     [{"calcular_frete": {"cep_origem": ["74000000"], "cep_destino": ["20040002"], "peso_kg": [1]}},
      {"calcular_frete": {"cep_origem": ["74000000"], "cep_destino": ["20040002"], "peso_kg": [5]}}]),
    ("paralelas_06", "Busque voos de GYN para GRU em 2026-12-01 e em 2026-12-08.", ["buscar_voos"],
     [{"buscar_voos": {"origem": ["GYN"], "destino": ["GRU"], "data": ["2026-12-01"], "passageiros": ["", 1]}},
      {"buscar_voos": {"origem": ["GYN"], "destino": ["GRU"], "data": ["2026-12-08"], "passageiros": ["", 1]}}]),
    ("paralelas_07", "Converta 5 km e 21 km para milhas.", ["converter_unidade"],
     [{"converter_unidade": {"valor": [5], "de": ["km"], "para": ["milha"]}}, {"converter_unidade": {"valor": [21], "de": ["km"], "para": ["milha"]}}]),
    ("paralelas_08", "Quais são os endereços dos CEPs 74605010 e 01310100?", ["consultar_cep"],
     [{"consultar_cep": {"cep": ["74605010"]}}, {"consultar_cep": {"cep": ["01310100"]}}]),
    ("paralelas_09", "Crie dois lembretes: 'pagar o boleto' em 2026-11-10 às 08:00 e 'ligar para o banco' em 2026-11-11 às 10:00.", ["criar_lembrete"],
     [{"criar_lembrete": {"texto": ["pagar o boleto", "pagar boleto"], "data": ["2026-11-10"], "hora": ["08:00"]}},
      {"criar_lembrete": {"texto": ["ligar para o banco", "ligar pro banco"], "data": ["2026-11-11"], "hora": ["10:00"]}}]),
    ("paralelas_10", "Quanto rendem R$ 1.000 a 1% ao mês em 6 meses e em 12 meses?", ["calcular_juros_compostos"],
     [{"calcular_juros_compostos": {"capital": [1000], "taxa_mensal": [1], "meses": [6]}},
      {"calcular_juros_compostos": {"capital": [1000], "taxa_mensal": [1], "meses": [12]}}]),

    # ---------- paralelas_multiplas: 3 ferramentas, chamadas a funções diferentes ----------
    ("paralelas_multiplas_01", "Como está o tempo em Lisboa e quanto valem 500 euros em reais?", ["consultar_clima", "converter_moeda", "buscar_voos"],
     [{"consultar_clima": {"cidade": LISBOA, "unidade": CEL}}, {"converter_moeda": {"valor": [500], "origem": ["EUR"], "destino": ["BRL"]}}]),
    ("paralelas_multiplas_02", "Qual a cotação da PETR4 e quais são as 5 notícias mais recentes sobre a Petrobras?", ["cotacao_acao", "buscar_noticias", "converter_moeda"],
     [{"cotacao_acao": {"ticker": ["PETR4"]}}, {"buscar_noticias": {"tema": ["Petrobras", "a Petrobras"], "quantidade": ["", 5]}}]),
    ("paralelas_multiplas_03", "Qual o endereço do CEP 74605010 e quanto custa mandar 3 kg de lá para o CEP 01310100?", ["consultar_cep", "calcular_frete", "consultar_clima"],
     [{"consultar_cep": {"cep": ["74605010"]}}, {"calcular_frete": {"cep_origem": ["74605010"], "cep_destino": ["01310100"], "peso_kg": [3]}}]),
    ("paralelas_multiplas_04", "Traduza 'boa viagem' para o italiano e me lembre em 2026-12-19 às 07:00 de fazer as malas.", ["traduzir_texto", "criar_lembrete", "buscar_voos"],
     [{"traduzir_texto": {"texto": ["boa viagem"], "idioma": ["it"]}}, {"criar_lembrete": {"texto": ["*"], "data": ["2026-12-19"], "hora": ["07:00"]}}]),
    ("paralelas_multiplas_05", "Busque voos de BSB para REC em 2026-12-23 e me diga como está o tempo em Recife.", ["buscar_voos", "consultar_clima", "consultar_cep"],
     [{"buscar_voos": {"origem": ["BSB"], "destino": ["REC"], "data": ["2026-12-23"], "passageiros": ["", 1]}},
      {"consultar_clima": {"cidade": RECIFE, "unidade": CEL}}]),
    ("paralelas_multiplas_06", "Converta 20 litros para galões e 50 dólares para reais.", ["converter_unidade", "converter_moeda", "calcular_frete"],
     [{"converter_unidade": {"valor": [20], "de": ["litro"], "para": ["galao"]}}, {"converter_moeda": {"valor": [50], "origem": ["USD"], "destino": ["BRL"]}}]),
    ("paralelas_multiplas_07", "Me sugira uma receita com mandioca e traga 2 notícias sobre a safra de mandioca.", ["buscar_receitas", "buscar_noticias", "traduzir_texto"],
     [{"buscar_receitas": {"ingrediente": ["mandioca"], "tempo_max_min": [""]}},
      {"buscar_noticias": {"tema": ["safra de mandioca", "safra da mandioca", "a safra de mandioca"], "quantidade": [2]}}]),
    ("paralelas_multiplas_08", "Qual a cotação de ITUB4 e de BBDC4, e quanto valem 1.000 dólares em reais?", ["cotacao_acao", "converter_moeda", "buscar_noticias"],
     [{"cotacao_acao": {"ticker": ["ITUB4"]}}, {"cotacao_acao": {"ticker": ["BBDC4"]}},
      {"converter_moeda": {"valor": [1000], "origem": ["USD"], "destino": ["BRL"]}}]),
    ("paralelas_multiplas_09", "Como está o tempo em Goiânia, em fahrenheit, e qual o endereço do CEP 74605010?", ["consultar_clima", "consultar_cep", "calcular_frete"],
     [{"consultar_clima": {"cidade": GOIANIA, "unidade": ["fahrenheit"]}}, {"consultar_cep": {"cep": ["74605010"]}}]),
    ("paralelas_multiplas_10", "Quanto rendem R$ 2.000 a 0,8% ao mês em 24 meses? E quanto valem 2.000 reais em dólares?", ["calcular_juros_compostos", "converter_moeda", "cotacao_acao"],
     [{"calcular_juros_compostos": {"capital": [2000], "taxa_mensal": [0.8], "meses": [24]}},
      {"converter_moeda": {"valor": [2000], "origem": ["BRL"], "destino": ["USD"]}}]),

    # ---------- irrelevancia: nenhuma ferramenta serve, o certo é não chamar ----------
    ("irrelevancia_01", "Qual é a capital da Austrália?", ["consultar_clima", "buscar_voos"], []),
    ("irrelevancia_02", "Converta 30 graus Celsius para Fahrenheit.", ["converter_unidade", "converter_moeda"], []),
    ("irrelevancia_03", "Quanto é 15% de 240?", ["calcular_juros_compostos", "converter_moeda"], []),
    ("irrelevancia_04", "Escreva um poema curto sobre o cerrado.", ["traduzir_texto", "buscar_receitas"], []),
    ("irrelevancia_05", "Quem escreveu Grande Sertão: Veredas?", ["cotacao_acao", "consultar_cep"], []),
    ("irrelevancia_06", "Qual é a raiz quadrada de 144?", ["converter_unidade", "calcular_juros_compostos"], []),
    ("irrelevancia_07", "Reserve um hotel em Lisboa para o fim de dezembro.", ["buscar_voos", "consultar_clima"], []),
    ("irrelevancia_08", "Envie um e-mail para a Elisa avisando que os slides estão prontos.", ["criar_lembrete", "traduzir_texto"], []),
    ("irrelevancia_09", "Quantos dias faltam para o Natal?", ["criar_lembrete", "consultar_clima"], []),
    ("irrelevancia_10", "Qual é o saldo da minha conta bancária?", ["cotacao_acao", "converter_moeda", "calcular_juros_compostos"], []),

    # ---------- falta_info: a ferramenta serve, mas falta um dado obrigatório ----------
    ("falta_info_01", "Me lembre de beber água.", ["criar_lembrete"], []),
    ("falta_info_02", "Quanto custa o frete de 2 kg para o CEP 01310100?", ["calcular_frete"], []),
    ("falta_info_03", "Busque voos para Salvador em 2026-12-28.", ["buscar_voos"], []),
    ("falta_info_04", "Qual a cotação da ação?", ["cotacao_acao"], []),
    ("falta_info_05", "Traduza 'bom trabalho'.", ["traduzir_texto"], []),
    ("falta_info_06", "Quanto rende minha aplicação de R$ 3.000 em 10 meses?", ["calcular_juros_compostos"], []),
    ("falta_info_07", "Como está o tempo?", ["consultar_clima"], []),
    ("falta_info_08", "Qual é o endereço desse CEP?", ["consultar_cep"], []),
    ("falta_info_09", "Converta 12 para milhas.", ["converter_unidade"], []),
    ("falta_info_10", "Crie um lembrete para sexta às 15h.", ["criar_lembrete"], []),
]

CASOS = []
for cid, pedido, ferramentas, esperado in CASOS_PT:
    CASOS.append(dict(id=cid, categoria=cid.rsplit("_", 1)[0], fonte="pt", pedido=pedido,
                      ferramentas=ferramentas, defs={n: TOOLS[n] for n in ferramentas}, esperado=esperado))
print(len(CASOS), "casos em português")

### 2b. Casos do próprio BFCL (ponte para a Fase 2)

Para que este artefato já seja um primeiro passo da reprodução, misturamos uma amostra pequena do **BFCL v3** (em inglês) e passamos pelos mesmos passos: geração, parsing, checker, métricas. Só entram casos cujos parâmetros são escalares ou listas de escalares (o nosso checker não trata dicionários aninhados).

Como o BFCL faz para modelos que não aceitam ponto no nome, `math.factorial` vira `math_factorial`. Se o download falhar, o notebook segue só com os casos em português. O hash de cada arquivo baixado entra no config: na Semana 6 a revisão do dataset ficou fora do hash; aqui não.

In [ ]:
USAR_BFCL = True
N_BFCL = {"simple": 20, "multiple": 10, "parallel": 10, "irrelevance": 10}
BFCL_HASHES = {}

if USAR_BFCL:
    try:
        from huggingface_hub import hf_hub_download
        REPO_BFCL = "gorilla-llm/Berkeley-Function-Calling-Leaderboard"
        TIPOS_BFCL = {"dict": "object", "float": "number", "tuple": "array"}
        ESCALARES = ("string", "integer", "float", "boolean")
        rng_bfcl = random.Random(0)
        for cat, n_amostra in N_BFCL.items():
            caminho = hf_hub_download(REPO_BFCL, f"BFCL_v3_{cat}.json", repo_type="dataset")
            bruto = open(caminho, encoding="utf-8").read()
            BFCL_HASHES[cat] = hashlib.sha256(bruto.encode()).hexdigest()[:16]
            entradas = [json.loads(l) for l in bruto.splitlines() if l.strip()]
            respostas = {}
            if cat != "irrelevance":
                cam_r = hf_hub_download(REPO_BFCL, f"possible_answer/BFCL_v3_{cat}.json", repo_type="dataset")
                for l in open(cam_r, encoding="utf-8"):
                    if l.strip():
                        r = json.loads(l)
                        respostas[r["id"]] = r["ground_truth"]
            elegiveis = []
            for e in entradas:
                ok = len(e["question"]) == 1 and (cat == "irrelevance" or e["id"] in respostas)
                for f in e["function"]:
                    for p in f["parameters"].get("properties", {}).values():
                        if p.get("type") not in ESCALARES + ("array",):
                            ok = False
                        if p.get("type") == "array" and p.get("items", {}).get("type") not in ESCALARES:
                            ok = False
                if ok:
                    elegiveis.append(e)
            for e in rng_bfcl.sample(elegiveis, min(n_amostra, len(elegiveis))):
                defs = {}
                for f in e["function"]:
                    params = copy.deepcopy(f["parameters"])
                    params["type"] = "object"
                    for p in params.get("properties", {}).values():
                        p["type"] = TIPOS_BFCL.get(p["type"], p["type"])
                        if "items" in p:
                            p["items"]["type"] = TIPOS_BFCL.get(p["items"]["type"], p["items"]["type"])
                    nome = f["name"].replace(".", "_")
                    defs[nome] = {"name": nome, "description": f.get("description", ""), "parameters": params}
                esperado = [{k.replace(".", "_"): v for k, v in gt.items()} for gt in respostas.get(e["id"], [])]
                msgs_usuario = [m for m in e["question"][0] if m["role"] == "user"]
                CASOS.append(dict(id="bfcl_" + e["id"], categoria="bfcl_" + cat, fonte="bfcl", pedido=msgs_usuario[-1]["content"],
                                  ferramentas=list(defs), defs=defs, esperado=esperado))
            print(f"BFCL {cat}: {len(elegiveis)} elegíveis, {min(n_amostra, len(elegiveis))} sorteados | hash {BFCL_HASHES[cat]}")
    except Exception as ex:
        print("Não foi possível carregar o BFCL; seguindo só com os casos em português:", repr(ex))
print(len(CASOS), "casos no total")

In [ ]:
# Checagens do próprio gabarito: um benchmark com gabarito errado mede o erro do gabarito.
problemas = []
for caso in CASOS:
    for chamada in caso["esperado"]:
        for nome, params in chamada.items():
            if nome not in caso["defs"]:
                problemas.append((caso["id"], "função esperada não oferecida", nome))
                continue
            schema = caso["defs"][nome]["parameters"]
            for k, aceitos in params.items():
                if k not in schema.get("properties", {}):
                    problemas.append((caso["id"], "parâmetro fora do schema", k))
                elif k in schema.get("required", []) and "" in aceitos:
                    problemas.append((caso["id"], "obrigatório marcado como opcional", k))
                else:
                    for a in aceitos:
                        if isinstance(a, str) and a in ("", "*"):
                            continue
                        if list(Draft202012Validator(schema["properties"][k]).iter_errors(a)):
                            problemas.append((caso["id"], f"valor aceito que o próprio schema rejeita ({k})", repr(a)))
            for k in schema.get("required", []):
                if k not in params:
                    problemas.append((caso["id"], "obrigatório ausente do gabarito", k))
problemas_pt = [p for p in problemas if not p[0].startswith("bfcl_")]
print("problemas no gabarito em português:", problemas_pt)
print("avisos no gabarito do BFCL:", len(problemas) - len(problemas_pt), problemas[len(problemas_pt):][:5])
assert not problemas_pt, "corrija o gabarito antes de seguir"

# Hash do dataset (antes de qualquer campo derivado)
DATASET_HASH = hashlib.sha256(json.dumps(
    [{k: c[k] for k in ("id", "categoria", "pedido", "ferramentas", "defs", "esperado")} for c in CASOS],
    sort_keys=True, ensure_ascii=False).encode()).hexdigest()[:16]

# Gabarito pré-normalizado: cada valor aceito vira uma "forma canônica" comparável
# (texto normalizado, número como float, booleano, lista de canônicos)
for caso in CASOS:
    caso["gab"] = []
    for chamada in caso["esperado"]:
        for nome, params in chamada.items():
            g = {"nome": nome, "args": {}}
            for k, aceitos in params.items():
                canon = set()
                for a in aceitos:
                    if isinstance(a, str) and a in ("", "*"):
                        continue
                    if isinstance(a, bool):
                        canon.add(("b", a))
                    elif isinstance(a, (int, float)):
                        canon.add(("n", float(a)))
                    elif isinstance(a, str):
                        canon.add(("s", RE_NAO_ALNUM.sub("", unicodedata.normalize("NFKD", a).encode("ascii", "ignore").decode().lower())))
                    elif isinstance(a, list):
                        elems = []
                        for x in a:
                            if isinstance(x, bool):
                                elems.append(("b", x))
                            elif isinstance(x, (int, float)):
                                elems.append(("n", float(x)))
                            elif isinstance(x, str):
                                elems.append(("s", RE_NAO_ALNUM.sub("", unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode().lower())))
                            else:
                                elems.append(("o", json.dumps(x, sort_keys=True)))
                        canon.add(("l", tuple(elems)))
                    else:
                        canon.add(("o", json.dumps(a, sort_keys=True)))
                g["args"][k] = {"opcional": any(isinstance(a, str) and a == "" for a in aceitos),
                                "qualquer": any(isinstance(a, str) and a == "*" for a in aceitos), "canon": canon}
            caso["gab"].append(g)

CASO = {c["id"]: c for c in CASOS}
display(pd.crosstab(pd.Series([c["categoria"] for c in CASOS], name="categoria"),
                    pd.Series([c["fonte"] for c in CASOS], name="fonte")))
print("dataset_hash:", DATASET_HASH)

## 3. Configuração congelada e o prompt que o modelo vê

Tudo que define o experimento fica em `CONFIG`, e o hash dele entra em cada arquivo salvo.

Atenção a um detalhe que muda resultados sem ninguém perceber: o `generation_config.json` do Qwen2.5-Instruct **liga amostragem por padrão** (`do_sample=true`, `temperature=0.7`, `top_p=0.8`, `top_k=20`, `repetition_penalty=1.1` no 0.5B). Se não fixarmos a decodificação, duas execuções do mesmo notebook dão números diferentes.

- **Single-turn:** decodificação gulosa (determinística), uma execução por modelo.
- **Multi-turn:** amostragem com 5 seeds, de propósito: queremos medir **consistência** (pass^k). Mesmos parâmetros para os dois modelos. Tiramos a `repetition_penalty` (1.0) porque chamadas paralelas repetem as mesmas chaves JSON.

In [ ]:
CONFIG = dict(
    modelos=["Qwen/Qwen2.5-0.5B-Instruct", "Qwen/Qwen2.5-1.5B-Instruct"],  # opcional: "Qwen/Qwen2.5-3B-Instruct" (use dtype "float16")
    dtype="float32",
    sistema="Você é um assistente que pode chamar ferramentas para ajudar o usuário. Hoje é quarta-feira, 2026-11-04. Responda em português.",
    sistema_bfcl="You are a helpful assistant that can call tools.",
    gen_single=dict(do_sample=False, temperature=None, top_p=None, top_k=None, repetition_penalty=1.0, max_new_tokens=256),
    lote=12,
    sistema_agente=("Você é o assistente de escritório da equipe de benchmarks. Hoje é quarta-feira, 2026-11-04. "
                    "Use as ferramentas para consultar e alterar a agenda e para enviar e-mails. Regras: "
                    "(1) antes de criar um evento, consulte a agenda do dia para verificar se o horário está livre; "
                    "(2) se faltar alguma informação necessária, pergunte ao usuário antes de agir; "
                    "(3) nunca invente e-mails ou identificadores: obtenha-os com as ferramentas. Responda em português."),
    gen_multi=dict(do_sample=True, temperature=0.7, top_p=0.8, top_k=20, repetition_penalty=1.0, max_new_tokens=256),
    seeds_multi=[0, 1, 2, 3, 4],
    max_passos_por_turno=6,
    dataset_hash=DATASET_HASH,
    bfcl=dict(usar=USAR_BFCL, n=N_BFCL, hashes=BFCL_HASHES),
)
CONFIG_HASH = hashlib.sha256(json.dumps(CONFIG, sort_keys=True, ensure_ascii=False).encode()).hexdigest()[:16]
print("config_hash:", CONFIG_HASH)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained(CONFIG["modelos"][0])
tok.padding_side = "left"   # geração em lote: o texto novo começa na mesma posição para todos

# os modelos comparados precisam usar o mesmo chat template (senão o prompt muda junto com o modelo)
for nome_modelo in CONFIG["modelos"][1:]:
    outro = AutoTokenizer.from_pretrained(nome_modelo)
    print(nome_modelo, "| mesmo chat template:", outro.chat_template == tok.chat_template)

caso = CASO["simples_01"]
ferramentas = [{"type": "function", "function": caso["defs"][n]} for n in caso["ferramentas"]]
mensagens = [{"role": "system", "content": CONFIG["sistema"]}, {"role": "user", "content": caso["pedido"]}]
prompt = tok.apply_chat_template(mensagens, tools=ferramentas, add_generation_prompt=True, tokenize=False)
print(prompt)
print("tokens no prompt:", len(tok(prompt)["input_ids"]))
# As ferramentas viram TEXTO no prompt de sistema. A "chamada" também é texto: o que estiver
# entre <tool_call> e </tool_call>. Quem transforma esse texto em ação é o parser + executor.

## 4. Geração single-turn

Uma passada gulosa por modelo, em lotes. A **saída bruta** fica salva: o checker pode mudar depois sem precisar gerar de novo (e outra pessoa pode reavaliar as mesmas saídas).

In [ ]:
saidas = {}
for nome_modelo in CONFIG["modelos"]:
    arq = f"{OUT}/single_{nome_modelo.split('/')[-1]}.json"
    if os.path.exists(arq):
        reg = json.load(open(arq))
        if reg["config_hash"] == CONFIG_HASH:
            saidas[nome_modelo] = reg["saidas"]
            print("reaproveitado:", arq)
            continue
        print("config mudou desde a última execução; gerando de novo:", arq)

    modelo = AutoModelForCausalLM.from_pretrained(nome_modelo, dtype=getattr(torch, CONFIG["dtype"]),
                                                  device_map={"": 0} if DEV == "cuda" else None)
    modelo.eval()
    prompts, ids = [], []
    for caso in CASOS:
        sistema = CONFIG["sistema_bfcl"] if caso["fonte"] == "bfcl" else CONFIG["sistema"]
        ferramentas = [{"type": "function", "function": caso["defs"][n]} for n in caso["ferramentas"]]
        mensagens = [{"role": "system", "content": sistema}, {"role": "user", "content": caso["pedido"]}]
        prompts.append(tok.apply_chat_template(mensagens, tools=ferramentas, add_generation_prompt=True, tokenize=False))
        ids.append(caso["id"])

    t0 = datetime.datetime.now()
    textos = []
    for i in range(0, len(prompts), CONFIG["lote"]):
        enc = tok(prompts[i:i + CONFIG["lote"]], return_tensors="pt", padding=True).to(modelo.device)
        with torch.no_grad():
            out = modelo.generate(**enc, **CONFIG["gen_single"], pad_token_id=tok.pad_token_id)
        textos += tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=False)
    textos = [t.replace("<|im_end|>", "").replace("<|endoftext|>", "").strip() for t in textos]
    segundos = (datetime.datetime.now() - t0).total_seconds()

    saidas[nome_modelo] = dict(zip(ids, textos))
    json.dump(dict(tipo="single", modelo=nome_modelo, config_hash=CONFIG_HASH, dataset_hash=DATASET_HASH, env=ENV,
                   timestamp=datetime.datetime.now(datetime.timezone.utc).isoformat(), segundos=segundos,
                   saidas=saidas[nome_modelo]), open(arq, "w"), ensure_ascii=False, indent=1)
    print(f"{nome_modelo}: {len(textos)} saídas em {segundos:.0f} s -> {arq}")
    del modelo
    gc.collect()
    if DEV == "cuda":
        torch.cuda.empty_cache()

print("\nExemplo de saída bruta (simples_01):")
for nome_modelo in saidas:
    print(f"--- {nome_modelo}\n{saidas[nome_modelo]['simples_01']}")

## 5. Parsing e validação

Três portões, cada um pega um tipo de erro diferente:

1. **Parsing:** o texto vira JSON com `name` e `arguments`?
2. **Schema:** os argumentos respeitam o JSON Schema (tipos, `enum`, `pattern`, obrigatórios)?
3. **Checker (seção 6):** os valores são os que o usuário pediu?

Dois parsers, para mostrar que **ler a saída é parte do método**:
- **estrito:** só vale o que está entre `<tool_call>` e `</tool_call>` (formato que o template do Qwen pede);
- **tolerante:** se não houver tags, aceita também JSON solto ou em bloco de código.

Antes das saídas dos modelos, entram algumas **saídas escritas à mão** (`exemplos_manuais`) para testar o checker: cada uma produz um tipo de erro conhecido.

In [ ]:
EXEMPLOS_MANUAIS = [
    ("simples_02", "chamada correta",
     '<tool_call>\n{"name": "converter_moeda", "arguments": {"valor": 250, "origem": "USD", "destino": "BRL"}}\n</tool_call>'),
    ("multiplas_03", "leu 1.200 como 1,2",
     '<tool_call>\n{"name": "converter_moeda", "arguments": {"valor": 1.2, "origem": "EUR", "destino": "BRL"}}\n</tool_call>'),
    ("simples_03", "CEP com traço",
     '<tool_call>\n{"name": "calcular_frete", "arguments": {"cep_origem": "74605-010", "cep_destino": "01310-100", "peso_kg": 2.5}}\n</tool_call>'),
    ("multiplas_06", "tratou libra como moeda",
     '<tool_call>\n{"name": "converter_moeda", "arguments": {"valor": 10, "origem": "GBP", "destino": "BRL"}}\n</tool_call>'),
    ("simples_04", "número escrito como texto",
     '<tool_call>\n{"name": "buscar_voos", "arguments": {"origem": "GYN", "destino": "BSB", "data": "2026-12-20", "passageiros": "2"}}\n</tool_call>'),
    ("simples_09", "plural fora do enum",
     '<tool_call>\n{"name": "converter_unidade", "arguments": {"valor": 42, "de": "km", "para": "milhas"}}\n</tool_call>'),
    ("paralelas_01", "esqueceu uma cidade",
     '<tool_call>\n{"name": "consultar_clima", "arguments": {"cidade": "Goiânia"}}\n</tool_call>\n'
     '<tool_call>\n{"name": "consultar_clima", "arguments": {"cidade": "Brasília"}}\n</tool_call>'),
    ("irrelevancia_02", "forçou uma ferramenta que não serve",
     '<tool_call>\n{"name": "converter_unidade", "arguments": {"valor": 30, "de": "celsius", "para": "fahrenheit"}}\n</tool_call>'),
    ("falta_info_01", "inventou data e hora",
     '<tool_call>\n{"name": "criar_lembrete", "arguments": {"texto": "beber água", "data": "2026-11-04", "hora": "10:00"}}\n</tool_call>'),
    ("simples_05", "JSON quebrado (falta uma chave)",
     '<tool_call>\n{"name": "cotacao_acao", "arguments": {"ticker": "PETR4"}\n</tool_call>'),
    ("simples_05", "JSON certo, mas sem as tags",
     '```json\n{"name": "cotacao_acao", "arguments": {"ticker": "PETR4"}}\n```'),
    ("simples_08", "inventou um filtro que ninguém pediu",
     '<tool_call>\n{"name": "buscar_receitas", "arguments": {"ingrediente": "abóbora", "tempo_max_min": 30}}\n</tool_call>'),
    ("simples_01", "parâmetro que não existe",
     '<tool_call>\n{"name": "consultar_clima", "arguments": {"cidade": "Goiânia", "pais": "Brasil"}}\n</tool_call>'),
    ("simples_07", "faltou um argumento",
     '<tool_call>\n{"name": "calcular_juros_compostos", "arguments": {"capital": 5000, "taxa_mensal": 1.2}}\n</tool_call>'),
    ("multiplas_01", "função que não existe",
     '<tool_call>\n{"name": "previsao_do_tempo", "arguments": {"cidade": "Recife"}}\n</tool_call>'),
    ("falta_info_07", "perguntou em vez de chamar",
     "De qual cidade você quer saber o tempo?"),
]
print(len(EXEMPLOS_MANUAIS), "saídas manuais")

In [ ]:
RE_BLOCO = re.compile(r"<tool_call>(.*?)</tool_call>", re.S)
RE_CERCA = re.compile(r"```(?:json)?\s*(.*?)```", re.S)

entradas_parser = [("exemplos_manuais", cid, rot, txt) for cid, rot, txt in EXEMPLOS_MANUAIS]
for nome_modelo, d in saidas.items():
    for cid, txt in d.items():
        entradas_parser.append((nome_modelo, cid, "", txt))

linhas = []
for nome_modelo, cid, rotulo, texto in entradas_parser:
    caso = CASO[cid]

    # ---- parser estrito: só o que está entre <tool_call> e </tool_call>
    blocos = RE_BLOCO.findall(texto)
    cham_e, blocos_ok = [], True
    for b in blocos:
        try:
            obj = json.loads(b.strip())
            args = obj.get("arguments")
            if isinstance(args, str):
                args = json.loads(args)
            if isinstance(obj.get("name"), str) and isinstance(args, dict):
                cham_e.append([obj["name"], args])
            else:
                blocos_ok = False
        except Exception:
            blocos_ok = False
    tag_aberta = texto.count("<tool_call>") > len(blocos)      # ex.: corte por max_new_tokens
    fmt_e = blocos_ok and not tag_aberta

    # ---- parser tolerante: além das tags, aceita JSON solto ou em bloco de código
    cham_t, fmt_t = [list(c) for c in cham_e], blocos_ok
    candidatos = []
    if not blocos:
        candidatos = RE_CERCA.findall(texto)
        if not candidatos and texto.strip()[:1] in ("{", "["):
            candidatos = [texto.strip()]
    if tag_aberta:
        candidatos.append(texto.split("<tool_call>")[-1])
    for cand in candidatos:
        try:
            obj = json.loads(cand.strip())
            for item in (obj if isinstance(obj, list) else [obj]):
                args = item.get("arguments", item.get("parameters"))
                if isinstance(args, str):
                    args = json.loads(args)
                if isinstance(item.get("name"), str) and isinstance(args, dict):
                    cham_t.append([item["name"], args])
                else:
                    fmt_t = False
        except Exception:
            fmt_t = False

    # ---- validação pelo JSON Schema (sobre o parsing estrito)
    erros_schema = []
    for nome, args in cham_e:
        if nome not in caso["defs"]:
            erros_schema.append(f"{nome}: função não oferecida")
        else:
            erros_schema += [f"{nome}: {e.message}" for e in Draft202012Validator(caso["defs"][nome]["parameters"]).iter_errors(args)]

    linhas.append(dict(modelo=nome_modelo, id=cid, rotulo=rotulo, categoria=caso["categoria"], fonte=caso["fonte"], texto=texto,
                       chamadas_estrito=cham_e, formato_estrito=fmt_e, chamadas_tolerante=cham_t, formato_tolerante=fmt_t,
                       schema_ok=(not erros_schema) if cham_e else None, erros_schema=erros_schema))
print(len(linhas), "linhas analisadas")

## 6. Checker AST e taxonomia de erros

O checker segue a ordem do BFCL e devolve **o primeiro erro** encontrado:

| Rótulo | Portão | Quando acontece |
|---|---|---|
| `formato_invalido` | parsing | há tags, mas o JSON não fecha ou não tem `name`/`arguments` |
| `nao_chamou` | seleção | deveria chamar e respondeu em texto |
| `chamada_indevida` | seleção | não deveria chamar (irrelevância, falta de informação) e chamou |
| `quantidade_errada` | seleção | número de chamadas diferente do esperado |
| `funcao_inexistente` | seleção | inventou uma função que não foi oferecida |
| `funcao_errada` | seleção | escolheu uma função oferecida, mas não a certa |
| `arg_inexistente` | argumentos | passou um parâmetro que não existe |
| `arg_faltando` | argumentos | omitiu um parâmetro obrigatório (ou um opcional que o pedido exigia) |
| `tipo_errado` | argumentos | tipo diferente do schema (ex.: `"2"` em vez de `2`) |
| `valor_errado` | argumentos | tipo certo, valor diferente do pedido |

Em chamadas paralelas, testamos todas as ordens possíveis e ficamos com a que casa mais chamadas; a avaliação é tudo-ou-nada.

In [ ]:
for linha in linhas:
    caso = CASO[linha["id"]]
    gab = caso["gab"]
    for modo in ("estrito", "tolerante"):
        chamadas, fmt = linha[f"chamadas_{modo}"], linha[f"formato_{modo}"]

        if not fmt:
            erro = "formato_invalido"
        elif not gab:
            erro = "ok" if not chamadas else "chamada_indevida"
        elif not chamadas:
            erro = "nao_chamou"
        elif len(chamadas) != len(gab):
            erro = "quantidade_errada"
        else:
            melhor = None
            for perm in permutations(range(len(chamadas))):
                erros_perm = []
                for (nome, args), g in zip([chamadas[i] for i in perm], gab):
                    e = "ok"
                    if nome not in caso["defs"]:
                        e = "funcao_inexistente"
                    elif nome != g["nome"]:
                        e = "funcao_errada"
                    else:
                        schema = caso["defs"][nome]["parameters"]
                        props = schema.get("properties", {})
                        for k in args:
                            if k not in props or k not in g["args"]:
                                e = "arg_inexistente"
                                break
                        if e == "ok":
                            for k in set(schema.get("required", [])) | {k for k, v in g["args"].items() if not v["opcional"]}:
                                if k not in args:
                                    e = "arg_faltando"
                                    break
                        if e == "ok":
                            for k, v in args.items():
                                tipo = props[k].get("type")
                                tipo_ok = (tipo is None
                                           or (tipo == "string" and isinstance(v, str))
                                           or (tipo == "integer" and isinstance(v, int) and not isinstance(v, bool))
                                           or (tipo == "number" and isinstance(v, (int, float)) and not isinstance(v, bool))
                                           or (tipo == "boolean" and isinstance(v, bool))
                                           or (tipo == "array" and isinstance(v, list))
                                           or (tipo == "object" and isinstance(v, dict)))
                                if not tipo_ok:
                                    e = "tipo_errado"
                                    break
                                # forma canônica do valor previsto (mesma regra usada no gabarito)
                                if isinstance(v, bool):
                                    vc = ("b", v)
                                elif isinstance(v, (int, float)):
                                    vc = ("n", float(v))
                                elif isinstance(v, str):
                                    vc = ("s", RE_NAO_ALNUM.sub("", unicodedata.normalize("NFKD", v).encode("ascii", "ignore").decode().lower()))
                                elif isinstance(v, list):
                                    elems = []
                                    for x in v:
                                        if isinstance(x, bool):
                                            elems.append(("b", x))
                                        elif isinstance(x, (int, float)):
                                            elems.append(("n", float(x)))
                                        elif isinstance(x, str):
                                            elems.append(("s", RE_NAO_ALNUM.sub("", unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode().lower())))
                                        else:
                                            elems.append(("o", json.dumps(x, sort_keys=True)))
                                    vc = ("l", tuple(elems))
                                else:
                                    vc = ("o", json.dumps(v, sort_keys=True))
                                aceito = vc in g["args"][k]["canon"] or (g["args"][k]["qualquer"] and vc[0] == "s" and vc[1] != "")
                                if not aceito:
                                    e = "valor_errado"
                                    break
                    erros_perm.append(e)
                if melhor is None or erros_perm.count("ok") > melhor.count("ok"):
                    melhor = erros_perm
            erro = next((e for e in melhor if e != "ok"), "ok")

        nomes_previstos = Counter(n for n, _ in chamadas)
        nomes_esperados = Counter(g["nome"] for g in gab)
        linha[f"erro_{modo}"] = erro
        linha[f"selecao_{modo}"] = bool(fmt and nomes_previstos == nomes_esperados)

dida = pd.DataFrame([l for l in linhas if l["modelo"] == "exemplos_manuais"])
dida["schema"] = dida["schema_ok"].map({True: "válido", False: "inválido", None: "sem chamada"})
pd.set_option("display.max_colwidth", 80)
display(dida[["id", "rotulo", "erro_estrito", "erro_tolerante", "schema"]])

Repare em três linhas da tabela acima:
- **"leu 1.200 como 1,2":** schema válido, checker reprova. Validar não é acertar.
- **"CEP com traço":** o checker aceita (a normalização remove o traço), mas o schema reprova (`pattern`). Os dois portões discordam: qual é o certo depende do que o sistema real faz com o argumento.
- **"JSON certo, mas sem as tags":** reprovado pelo parser estrito e aprovado pelo tolerante. O número final depende do parser, então o parser precisa ser declarado.

## 7. Métricas por categoria: seleção, argumentos, abstenção

Para cada modelo e categoria:
- **acurácia AST** (tudo certo), com IC 95% de Wilson: com 10 casos por categoria o intervalo é largo, e é isso que ele deve mostrar;
- **seleção:** chamou exatamente as funções certas (ou nenhuma, quando era para não chamar);
- **argumentos dado seleção:** entre os casos com a seleção certa, quantos acertaram todos os argumentos.

A decomposição `acerto = seleção × argumentos | seleção` separa as duas fontes de erro.

In [ ]:
df = pd.DataFrame([l for l in linhas if l["modelo"] != "exemplos_manuais"])
df["modelo"] = df["modelo"].str.split("/").str[-1]
MODELOS = [m.split("/")[-1] for m in CONFIG["modelos"]]
CATS_PT = ["simples", "multiplas", "paralelas", "paralelas_multiplas", "irrelevancia", "falta_info"]
CHAMAR = ["simples", "multiplas", "paralelas", "paralelas_multiplas"]
NAO_CHAMAR = ["irrelevancia", "falta_info"]

rows = []
for (fonte, nome_modelo, cat), g in df.groupby(["fonte", "modelo", "categoria"]):
    deve_chamar = bool(CASO[g["id"].iloc[0]]["esperado"])
    for modo in ("estrito", "tolerante"):
        n = len(g)
        k = int((g[f"erro_{modo}"] == "ok").sum())
        p = k / n
        z = 1.96
        centro = (p + z * z / (2 * n)) / (1 + z * z / n)
        meia = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
        sel = int(g[f"selecao_{modo}"].sum())
        rows.append(dict(fonte=fonte, modelo=nome_modelo, categoria=cat, modo=modo, n=n, acertos=k, acuracia=p,
                         ic_lo=centro - meia, ic_hi=centro + meia, selecao=sel / n,
                         args_dado_selecao=(k / sel if (sel and deve_chamar) else np.nan),
                         formato_invalido=float((g[f"erro_{modo}"] == "formato_invalido").mean())))
metricas = pd.DataFrame(rows)

m_pt = metricas[(metricas.fonte == "pt") & (metricas.modo == "estrito")]
print("Acurácia AST por categoria (parser estrito) e IC 95% de Wilson")
tabela_cat = m_pt.assign(txt=[f"{a:.2f} [{lo:.2f}, {hi:.2f}]" for a, lo, hi in zip(m_pt.acuracia, m_pt.ic_lo, m_pt.ic_hi)]
                         ).pivot(index="categoria", columns="modelo", values="txt").reindex(CATS_PT)[MODELOS]
display(tabela_cat)

resumo_single = []
for nome_modelo in MODELOS:
    g = df[(df.fonte == "pt") & (df.modelo == nome_modelo)]
    gch, gnc = g[g.categoria.isin(CHAMAR)], g[g.categoria.isin(NAO_CHAMAR)]
    sel = gch["selecao_estrito"].mean()
    ok = (gch["erro_estrito"] == "ok").mean()
    resumo_single.append(dict(modelo=nome_modelo, selecao=sel, argumentos_dado_selecao=(ok / sel if sel else np.nan),
                              acerto_quando_devia_chamar=ok, abstencao_correta=(gnc["erro_estrito"] == "ok").mean(),
                              acerto_geral_estrito=(g["erro_estrito"] == "ok").mean(),
                              acerto_geral_tolerante=(g["erro_tolerante"] == "ok").mean(),
                              schema_valido=g["schema_ok"].dropna().mean()))
resumo_single = pd.DataFrame(resumo_single)
print("\nUma métrica por capacidade (casos em português, parser estrito)")
display(resumo_single.round(3))

In [ ]:
# Figura 1: acurácia por categoria, um modelo por cor, barras de erro = IC 95% de Wilson
fig, ax = plt.subplots(figsize=(9, 4))
larg = 0.8 / len(MODELOS)
for j, nome_modelo in enumerate(MODELOS):
    s = m_pt[m_pt.modelo == nome_modelo].set_index("categoria").reindex(CATS_PT)
    x = np.arange(len(CATS_PT)) + (j - (len(MODELOS) - 1) / 2) * larg
    ax.bar(x, s.acuracia, width=larg * 0.92, color=CORES[j], label=nome_modelo,
           yerr=[s.acuracia - s.ic_lo, s.ic_hi - s.acuracia], capsize=3, error_kw=dict(lw=1, ecolor="#52514e"))
ax.set_xticks(np.arange(len(CATS_PT)))
ax.set_xticklabels([c.replace("_", "\n") for c in CATS_PT])
ax.set_ylim(0, 1.22); ax.set_yticks(np.arange(0, 1.01, 0.2))
ax.set_ylabel("acurácia AST (parser estrito)")
ax.set_title("Acerto por categoria, 10 casos cada; barras = IC 95% de Wilson", loc="left")
ax.legend(frameon=False, ncol=len(MODELOS), loc="upper right")
plt.tight_layout(); plt.savefig(f"{OUT}/fig_acuracia_por_categoria.png", dpi=200); plt.show()

# Figura 2: taxonomia de erros (o que deu errado, não só quanto)
ORDEM_ERROS = ["formato_invalido", "nao_chamou", "chamada_indevida", "quantidade_errada", "funcao_inexistente",
               "funcao_errada", "arg_inexistente", "arg_faltando", "tipo_errado", "valor_errado"]
cont = (df[(df.fonte == "pt") & (df.erro_estrito != "ok")].groupby(["modelo", "erro_estrito"]).size()
        .unstack(fill_value=0).reindex(index=MODELOS, columns=ORDEM_ERROS, fill_value=0))
fig, ax = plt.subplots(figsize=(8, 4.2))
alt = 0.8 / len(MODELOS)
for j, nome_modelo in enumerate(MODELOS):
    y = np.arange(len(ORDEM_ERROS)) + (j - (len(MODELOS) - 1) / 2) * alt
    ax.barh(y, cont.loc[nome_modelo], height=alt * 0.92, color=CORES[j], label=nome_modelo)
ax.set_yticks(np.arange(len(ORDEM_ERROS))); ax.set_yticklabels(ORDEM_ERROS); ax.invert_yaxis()
ax.set_xlabel("número de casos (de 60)")
ax.set_title("Primeiro erro encontrado pelo checker, por modelo", loc="left")
ax.legend(frameon=False, loc="lower right")
plt.tight_layout(); plt.savefig(f"{OUT}/fig_erros_por_tipo.png", dpi=200); plt.show()
display(cont)

In [ ]:
# Ler a saída é parte do método: quanto o número muda com o parser?
comp = (df[df.fonte == "pt"].assign(ok_estrito=df.erro_estrito == "ok", ok_tolerante=df.erro_tolerante == "ok")
        .groupby("modelo")[["ok_estrito", "ok_tolerante"]].mean().reindex(MODELOS))
comp["diferenca"] = comp.ok_tolerante - comp.ok_estrito
print("Acerto geral com cada parser (casos em português):")
display(comp.round(3))

# Validar não é acertar: cruzamento entre o JSON Schema e o checker (casos com pelo menos uma chamada)
com_chamada = df[(df.fonte == "pt") & df.schema_ok.notna()]
print("\nSchema × checker (parser estrito):")
display(pd.crosstab(com_chamada.schema_ok.map({True: "schema válido", False: "schema inválido"}),
                    com_chamada.erro_estrito.eq("ok").map({True: "checker: ok", False: "checker: erro"})))

# exemplos reais de cada tipo de erro, para a análise de erros na apresentação
print("\nUm exemplo real de cada tipo de erro (primeiro modelo):")
for erro in ORDEM_ERROS:
    ex = df[(df.fonte == "pt") & (df.modelo == MODELOS[0]) & (df.erro_estrito == erro)]
    if len(ex):
        r = ex.iloc[0]
        print(f"[{erro}] {r.id}: {CASO[r.id]['pedido']}\n    -> {r.texto[:220]!r}")

## 8. Multi-turn: do tool-calling ao comportamento agêntico

Agora o modelo **age**: cada chamada é executada num ambiente simulado (agenda + contatos + e-mails), o resultado volta como mensagem `tool`, e o modelo decide o próximo passo até responder ao usuário.

- **Multi-step:** um pedido, várias chamadas encadeadas (a saída de uma vira argumento da outra).
- **Multi-turn:** mais de uma mensagem do usuário (T5: o modelo deve perguntar o que falta e só agir depois).

**Como avaliamos (BFCL v3 e τ-bench):** pelo **estado final** do ambiente comparado ao estado esperado, mais checagens de trajetória: chamadas mínimas obrigatórias, nenhuma ação de escrita onde não devia, e, quando a tarefa é responder algo, palavras-chave na resposta final. O executor valida cada chamada pelo JSON Schema e devolve o erro ao modelo, que pode se corrigir.

**Consistência (τ-bench):** cada tarefa roda com 5 seeds. `pass^k` = probabilidade de **todas** as k tentativas darem certo: para uma tarefa com c sucessos em n tentativas, `C(c,k)/C(n,k)`, média entre as tarefas.

In [ ]:
EMAIL = r"^[^@\s]+@[^@\s]+\.[^@\s]+$"
FERR_ESC = {
    "buscar_contato": {"description": "Busca o e-mail de uma pessoa da equipe pelo nome.",
                       "parameters": {"type": "object", "properties": {"nome": {"type": "string", "description": "Nome da pessoa."}},
                                      "required": ["nome"]}},
    "listar_eventos": {"description": "Lista os eventos da agenda em uma data.",
                       "parameters": {"type": "object", "properties": {"data": DATA}, "required": ["data"]}},
    "criar_evento": {"description": "Cria um evento na agenda.",
                     "parameters": {"type": "object", "properties": {
                         "titulo": {"type": "string", "description": "Título do evento."}, "data": DATA, "hora": HORA,
                         "duracao_min": {"type": "integer", "minimum": 5, "description": "Duração em minutos."},
                         "participantes": {"type": "array", "items": {"type": "string", "pattern": EMAIL}, "description": "E-mails dos participantes."}},
                         "required": ["titulo", "data", "hora", "duracao_min", "participantes"]}},
    "cancelar_evento": {"description": "Cancela um evento da agenda pelo identificador.",
                        "parameters": {"type": "object", "properties": {"id_evento": {"type": "string", "description": "Identificador do evento, ex.: 'ev1'."}},
                                       "required": ["id_evento"]}},
    "enviar_email": {"description": "Envia um e-mail.",
                     "parameters": {"type": "object", "properties": {
                         "destinatario": {"type": "string", "pattern": EMAIL, "description": "E-mail do destinatário."},
                         "assunto": {"type": "string", "description": "Assunto."}, "corpo": {"type": "string", "description": "Texto do e-mail."}},
                         "required": ["destinatario", "assunto", "corpo"]}},
}
for nome in FERR_ESC:
    FERR_ESC[nome] = {"name": nome, **FERR_ESC[nome]}
TOOLS_ESC = [{"type": "function", "function": f} for f in FERR_ESC.values()]
ESCRITA = {"criar_evento", "cancelar_evento", "enviar_email"}

EQUIPE = {"Elisa": "elisa@exemplo.org", "Henrique": "henrique@exemplo.org", "Ana Clara": "anaclara@exemplo.org",
          "Gabriel": "gabriel@exemplo.org", "Jolie": "jolie@exemplo.org", "Mateus": "mateus@exemplo.org", "Lucca": "lucca@exemplo.org"}
TRIO = ["elisa@exemplo.org", "henrique@exemplo.org", "anaclara@exemplo.org"]
AGENDA0 = [
    dict(id="ev1", titulo="Daily", data="2026-11-06", hora="09:00", duracao_min=15, participantes=TRIO),
    dict(id="ev2", titulo="Retro da Semana 8", data="2026-11-06", hora="16:00", duracao_min=45, participantes=TRIO),
    dict(id="ev3", titulo="Reunião com o Lucca", data="2026-11-05", hora="10:00", duracao_min=60, participantes=["lucca@exemplo.org"]),
]
ESTADO0 = dict(contatos=EQUIPE, agenda=AGENDA0, emails=[], proximo_id=4)

TAREFAS = [
    dict(id="T1_encadear", habilidade="usar a saída de uma ferramenta como argumento de outra",
         turnos=["Marque uma reunião chamada 'Revisão do artefato' com a Elisa em 2026-11-09 às 14:00, com 30 minutos de duração."],
         agenda_final=AGENDA0 + [dict(titulo="Revisão do artefato", data="2026-11-09", hora="14:00", duracao_min=30, participantes=["elisa@exemplo.org"])],
         emails_finais=[], chamadas_obrigatorias=["buscar_contato", "criar_evento"]),
    dict(id="T2_cancelar_avisar", habilidade="quatro ações em sequência, com identificador lido da agenda",
         turnos=["Cancele a Daily de 2026-11-06 e avise o Henrique por e-mail que ela foi cancelada."],
         agenda_final=[AGENDA0[1], AGENDA0[2]], emails_finais=[dict(destinatario="henrique@exemplo.org")],
         chamadas_obrigatorias=["listar_eventos", "cancelar_evento", "buscar_contato", "enviar_email"]),
    dict(id="T3_consultar", habilidade="só leitura: responder sem alterar nada",
         turnos=["Quais reuniões eu tenho em 2026-11-06?"],
         agenda_final=AGENDA0, emails_finais=[], chamadas_obrigatorias=["listar_eventos"], resposta_contem_todas=["daily", "retro"]),
    dict(id="T4_paralelo", habilidade="chamadas paralelas e encadeadas",
         turnos=["Mande um e-mail para a Jolie e outro para o Mateus com o assunto 'Slides da Semana 8', avisando que os slides estão prontos."],
         agenda_final=AGENDA0, chamadas_obrigatorias=["buscar_contato", "enviar_email"],
         emails_finais=[dict(destinatario="jolie@exemplo.org", assunto_contem="slides"), dict(destinatario="mateus@exemplo.org", assunto_contem="slides")]),
    dict(id="T5_pedir_info", habilidade="multi-turn: perguntar o que falta antes de agir",
         turnos=["Marque uma reunião com o Gabriel.",
                 "Pode ser em 2026-11-10 às 11:00, por 60 minutos, com o título 'Planejamento da Fase 2'."],
         agenda_final=AGENDA0 + [dict(titulo="Planejamento da Fase 2", data="2026-11-10", hora="11:00", duracao_min=60, participantes=["gabriel@exemplo.org"])],
         emails_finais=[], chamadas_obrigatorias=["criar_evento"], sem_escrita_no_turno=[0]),
    dict(id="T6_nao_inventar", habilidade="lidar com um erro da ferramenta sem inventar dados",
         turnos=["Envie um e-mail para o Rafael dizendo que a apresentação é na sexta."],
         agenda_final=AGENDA0, emails_finais=[], chamadas_obrigatorias=["buscar_contato"],
         resposta_contem_alguma=["naoencontr", "naoexiste", "naoconsta", "naolocaliz", "naoachei", "naoha", "naotenho", "naofoipossivel"]),
    dict(id="T7_politica", habilidade="seguir a regra do sistema: checar a agenda antes de criar",
         turnos=["Marque 'Ensaio da apresentação' em 2026-11-05 às 10:00, por 60 minutos, com o Henrique, mas só se o horário estiver livre."],
         agenda_final=AGENDA0, emails_finais=[], chamadas_obrigatorias=["listar_eventos"],
         resposta_contem_alguma=["ocupad", "conflit", "naoestalivre", "indisponivel", "jaexiste", "jatem", "jaha", "lucca"]),
]
TAREFA = {t["id"]: t for t in TAREFAS}
for t in TAREFAS:
    print(f"{t['id']:20s} {t['habilidade']}")

In [ ]:
conversas = []
for nome_modelo in CONFIG["modelos"]:
    arq = f"{OUT}/multi_{nome_modelo.split('/')[-1]}.json"
    if os.path.exists(arq):
        reg = json.load(open(arq))
        if reg["config_hash"] == CONFIG_HASH:
            conversas += reg["conversas"]
            print("reaproveitado:", arq)
            continue

    modelo = AutoModelForCausalLM.from_pretrained(nome_modelo, dtype=getattr(torch, CONFIG["dtype"]),
                                                  device_map={"": 0} if DEV == "cuda" else None)
    modelo.eval()
    t0 = datetime.datetime.now()
    conversas_modelo = []
    for seed in CONFIG["seeds_multi"]:
        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
        lote = []
        for t in TAREFAS:
            lote.append(dict(modelo=nome_modelo, seed=seed, tarefa=t["id"], estado=copy.deepcopy(ESTADO0),
                             mensagens=[{"role": "system", "content": CONFIG["sistema_agente"]}, {"role": "user", "content": t["turnos"][0]}],
                             turno=0, passos_no_turno=0, n_passos=0, ativa=True, limite=False, n_invalidas=0, log=[], respostas=[]))

        while any(c["ativa"] for c in lote):
            ativas = [c for c in lote if c["ativa"]]
            prompts = [tok.apply_chat_template(c["mensagens"], tools=TOOLS_ESC, add_generation_prompt=True, tokenize=False) for c in ativas]
            textos = []
            for i in range(0, len(prompts), CONFIG["lote"]):
                enc = tok(prompts[i:i + CONFIG["lote"]], return_tensors="pt", padding=True).to(modelo.device)
                with torch.no_grad():
                    out = modelo.generate(**enc, **CONFIG["gen_multi"], pad_token_id=tok.pad_token_id)
                textos += tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=False)

            for c, texto in zip(ativas, textos):
                texto = texto.replace("<|im_end|>", "").replace("<|endoftext|>", "").strip()
                c["n_passos"] += 1
                c["passos_no_turno"] += 1
                t = TAREFA[c["tarefa"]]

                # parsing estrito: o harness só executa o que vier entre as tags
                blocos = RE_BLOCO.findall(texto)
                chamadas, invalidas = [], 0
                for b in blocos:
                    try:
                        obj = json.loads(b.strip())
                        args = obj.get("arguments")
                        if isinstance(args, str):
                            args = json.loads(args)
                        if isinstance(obj.get("name"), str) and isinstance(args, dict):
                            chamadas.append((obj["name"], args))
                        else:
                            invalidas += 1
                    except Exception:
                        invalidas += 1
                invalidas += texto.count("<tool_call>") - len(blocos)

                if not chamadas and not invalidas:
                    # resposta em texto: fim do turno do usuário
                    c["mensagens"].append({"role": "assistant", "content": texto})
                    c["respostas"].append(texto)
                    c["turno"] += 1
                    c["passos_no_turno"] = 0
                    if c["turno"] < len(t["turnos"]):
                        c["mensagens"].append({"role": "user", "content": t["turnos"][c["turno"]]})
                    else:
                        c["ativa"] = False
                    continue

                if chamadas:
                    c["mensagens"].append({"role": "assistant", "content": texto.split("<tool_call>")[0].strip(),
                                           "tool_calls": [{"type": "function", "function": {"name": n, "arguments": a}} for n, a in chamadas]})
                else:
                    c["mensagens"].append({"role": "assistant", "content": texto})

                # executor: valida pelo schema e aplica a ação no estado da conversa
                for nome, args in chamadas:
                    est = c["estado"]
                    if nome not in FERR_ESC:
                        resultado = {"erro": f"função inexistente: {nome}"}
                        c["n_invalidas"] += 1
                    elif list(Draft202012Validator(FERR_ESC[nome]["parameters"]).iter_errors(args)):
                        msgs = [e.message for e in Draft202012Validator(FERR_ESC[nome]["parameters"]).iter_errors(args)]
                        resultado = {"erro": "argumentos inválidos: " + "; ".join(msgs[:2])}
                        c["n_invalidas"] += 1
                    elif nome == "buscar_contato":
                        alvo = RE_NAO_ALNUM.sub("", unicodedata.normalize("NFKD", args["nome"]).encode("ascii", "ignore").decode().lower())
                        achados = []
                        for pessoa, email in est["contatos"].items():
                            chave = RE_NAO_ALNUM.sub("", unicodedata.normalize("NFKD", pessoa).encode("ascii", "ignore").decode().lower())
                            if alvo and (alvo in chave or chave in alvo):
                                achados.append({"nome": pessoa, "email": email})
                        resultado = achados[0] if len(achados) == 1 else ({"erro": "contato não encontrado"} if not achados else {"resultados": achados})
                    elif nome == "listar_eventos":
                        resultado = {"data": args["data"], "eventos": [{k: ev[k] for k in ("id", "titulo", "hora", "duracao_min", "participantes")}
                                                                      for ev in est["agenda"] if ev["data"] == args["data"]]}
                    elif nome == "criar_evento":
                        novo = dict(id=f"ev{est['proximo_id']}", titulo=args["titulo"], data=args["data"], hora=args["hora"],
                                    duracao_min=args["duracao_min"], participantes=args["participantes"])
                        est["proximo_id"] += 1
                        est["agenda"].append(novo)
                        resultado = {"status": "criado", "id": novo["id"]}
                    elif nome == "cancelar_evento":
                        alvo = [ev for ev in est["agenda"] if ev["id"] == args["id_evento"]]
                        if alvo:
                            est["agenda"].remove(alvo[0])
                            resultado = {"status": "cancelado", "id": args["id_evento"]}
                        else:
                            resultado = {"erro": "evento não encontrado"}
                    else:   # enviar_email
                        est["emails"].append(dict(args))
                        resultado = {"status": "enviado"}
                    c["log"].append(dict(turno=c["turno"], nome=nome, args=args, resultado=resultado))
                    c["mensagens"].append({"role": "tool", "name": nome, "content": json.dumps(resultado, ensure_ascii=False)})
                for _ in range(invalidas):
                    c["n_invalidas"] += 1
                    c["log"].append(dict(turno=c["turno"], nome="<mal_formatada>", args={}, resultado={"erro": "chamada mal formatada"}))
                    c["mensagens"].append({"role": "tool", "content": json.dumps({"erro": "chamada mal formatada: use JSON válido com name e arguments"}, ensure_ascii=False)})

                if c["passos_no_turno"] >= CONFIG["max_passos_por_turno"]:
                    c["limite"] = True
                    c["ativa"] = False
        conversas_modelo += lote

    for c in conversas_modelo:
        del c["ativa"]
    json.dump(dict(tipo="multi", modelo=nome_modelo, config_hash=CONFIG_HASH, env=ENV,
                   timestamp=datetime.datetime.now(datetime.timezone.utc).isoformat(),
                   segundos=(datetime.datetime.now() - t0).total_seconds(), conversas=conversas_modelo),
              open(arq, "w"), ensure_ascii=False, indent=1)
    print(f"{nome_modelo}: {len(conversas_modelo)} conversas em {(datetime.datetime.now() - t0).total_seconds():.0f} s -> {arq}")
    conversas += conversas_modelo
    del modelo
    gc.collect()
    if DEV == "cuda":
        torch.cuda.empty_cache()
print(len(conversas), "conversas")

In [ ]:
avaliacoes = []
for c in conversas:
    t = TAREFA[c["tarefa"]]
    est = c["estado"]

    # 1) estado final da agenda (ignora o identificador; título comparado por "contém")
    agenda_ok = len(est["agenda"]) == len(t["agenda_final"])
    usados = set()
    for esp in t["agenda_final"]:
        tit_esp = RE_NAO_ALNUM.sub("", unicodedata.normalize("NFKD", esp["titulo"]).encode("ascii", "ignore").decode().lower())
        achou = False
        for j, ev in enumerate(est["agenda"]):
            tit = RE_NAO_ALNUM.sub("", unicodedata.normalize("NFKD", str(ev["titulo"])).encode("ascii", "ignore").decode().lower())
            if (j not in usados and tit_esp in tit and ev["data"] == esp["data"] and ev["hora"] == esp["hora"]
                    and ev["duracao_min"] == esp["duracao_min"] and sorted(ev["participantes"]) == sorted(esp["participantes"])):
                usados.add(j)
                achou = True
                break
        agenda_ok = agenda_ok and achou

    # 2) e-mails enviados
    emails_ok = len(est["emails"]) == len(t["emails_finais"])
    usados = set()
    for esp in t["emails_finais"]:
        achou = False
        for j, em in enumerate(est["emails"]):
            assunto = RE_NAO_ALNUM.sub("", unicodedata.normalize("NFKD", str(em.get("assunto", ""))).encode("ascii", "ignore").decode().lower())
            if j not in usados and em["destinatario"].lower() == esp["destinatario"] and esp.get("assunto_contem", "") in assunto:
                usados.add(j)
                achou = True
                break
        emails_ok = emails_ok and achou

    # 3) trajetória: chamadas mínimas, nenhuma escrita onde não devia, resposta final
    nomes_chamados = {l["nome"] for l in c["log"]}
    obrigatorias_ok = all(n in nomes_chamados for n in t["chamadas_obrigatorias"])
    sem_escrita_ok = not any(l["nome"] in ESCRITA and l["turno"] in t.get("sem_escrita_no_turno", []) and "erro" not in l["resultado"]
                             for l in c["log"])
    resposta = RE_NAO_ALNUM.sub("", unicodedata.normalize("NFKD", c["respostas"][-1] if c["respostas"] else "").encode("ascii", "ignore").decode().lower())
    resposta_ok = (all(p in resposta for p in t.get("resposta_contem_todas", []))
                   and (not t.get("resposta_contem_alguma") or any(p in resposta for p in t["resposta_contem_alguma"])))

    # política do sistema: todo evento criado com sucesso precisa de um listar_eventos da mesma data antes
    politica_ok, consultadas = None, set()
    for l in c["log"]:
        if l["nome"] == "listar_eventos" and "erro" not in l["resultado"]:
            consultadas.add(l["args"].get("data"))
        if l["nome"] == "criar_evento" and "erro" not in l["resultado"]:
            politica_ok = (politica_ok is not False) and (l["args"].get("data") in consultadas)

    sucesso = agenda_ok and emails_ok and obrigatorias_ok and sem_escrita_ok and resposta_ok and not c["limite"]
    if sucesso:
        motivo = "sucesso"
    elif c["limite"]:
        motivo = "limite_de_passos"
    elif not sem_escrita_ok:
        motivo = "agiu_sem_informacao"
    elif not (agenda_ok and emails_ok):
        motivo = "estado_final_errado"
    elif not obrigatorias_ok:
        motivo = "pulou_chamada_necessaria"
    else:
        motivo = "resposta_incompleta"
    avaliacoes.append(dict(modelo=c["modelo"].split("/")[-1], seed=c["seed"], tarefa=c["tarefa"], sucesso=sucesso, motivo=motivo,
                           passos=c["n_passos"], chamadas=len(c["log"]), invalidas=c["n_invalidas"], politica_ok=politica_ok))
aval = pd.DataFrame(avaliacoes)

print("Taxa de sucesso por tarefa (5 seeds cada)")
display(aval.pivot_table(index="tarefa", columns="modelo", values="sucesso", aggfunc="mean")[MODELOS].round(2))
print("\nMotivo da falha (contagem)")
display(pd.crosstab(aval.motivo, aval.modelo)[MODELOS] if len(aval) else None)
print("\nDiagnóstico da trajetória (média por conversa; política = conversas que criaram evento e consultaram a agenda antes)")
com_criacao = aval.dropna(subset=["politica_ok"])
display(aval.groupby("modelo")[["passos", "chamadas", "invalidas"]].mean().reindex(MODELOS).round(2)
        .assign(politica_cumprida=com_criacao.politica_ok.astype(float).groupby(com_criacao.modelo).mean().round(2)))

# pass^k (τ-bench): para cada tarefa, C(c, k) / C(n, k); depois a média entre tarefas
passk = []
n_seeds = len(CONFIG["seeds_multi"])
for nome_modelo in MODELOS:
    g = aval[aval.modelo == nome_modelo]
    for k in range(1, n_seeds + 1):
        valores = [math.comb(int(gt.sucesso.sum()), k) / math.comb(len(gt), k) for _, gt in g.groupby("tarefa")]
        passk.append(dict(modelo=nome_modelo, k=k, pass_k=float(np.mean(valores))))
passk = pd.DataFrame(passk)
print("\npass^k")
display(passk.pivot(index="k", columns="modelo", values="pass_k")[MODELOS].round(3))
print("Exemplo do cálculo: tarefa com 3 sucessos em 5 -> pass^1 = 3/5 = 0.6 | pass^2 = C(3,2)/C(5,2) = 3/10 = 0.3 | pass^5 = 0")

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.8))
for j, nome_modelo in enumerate(MODELOS):
    s = passk[passk.modelo == nome_modelo]
    ax.plot(s.k, s.pass_k, marker="o", ms=7, lw=2, color=CORES[j], label=nome_modelo)
    ax.annotate(f"{s.pass_k.iloc[-1]:.2f}", (s.k.iloc[-1], s.pass_k.iloc[-1]), xytext=(6, 0),
                textcoords="offset points", va="center", fontsize=9, color="#52514e")
ax.set_xticks(range(1, n_seeds + 1)); ax.set_ylim(0, 1.02)
ax.set_xlabel("k (tentativas que precisam dar certo)"); ax.set_ylabel("pass^k")
ax.set_title("Acertar uma vez não basta: pass^k nas 7 tarefas multi-turn", loc="left")
ax.legend(frameon=False)
plt.tight_layout(); plt.savefig(f"{OUT}/fig_passk.png", dpi=200); plt.show()

fig, ax = plt.subplots(figsize=(8, 3.8))
tab_t = aval.pivot_table(index="tarefa", columns="modelo", values="sucesso", aggfunc="mean").reindex([t["id"] for t in TAREFAS])[MODELOS]
larg = 0.8 / len(MODELOS)
for j, nome_modelo in enumerate(MODELOS):
    x = np.arange(len(tab_t)) + (j - (len(MODELOS) - 1) / 2) * larg
    ax.bar(x, tab_t[nome_modelo], width=larg * 0.92, color=CORES[j], label=nome_modelo)
ax.set_xticks(np.arange(len(tab_t))); ax.set_xticklabels([i.split("_", 1)[0] + "\n" + i.split("_", 1)[1] for i in tab_t.index], fontsize=8)
ax.set_ylim(0, 1.22); ax.set_yticks(np.arange(0, 1.01, 0.2)); ax.set_ylabel("sucesso (5 seeds)")
ax.set_title("Sucesso por tarefa: estado final + trajetória", loc="left")
ax.legend(frameon=False, ncol=len(MODELOS))
plt.tight_layout(); plt.savefig(f"{OUT}/fig_sucesso_por_tarefa.png", dpi=200); plt.show()

In [ ]:
# Uma trajetória lida passo a passo (troque a tarefa, o modelo ou a seed para ver outras)
VER_MODELO, VER_TAREFA, VER_SEED = CONFIG["modelos"][0], "T2_cancelar_avisar", 0
c = [c for c in conversas if c["modelo"] == VER_MODELO and c["tarefa"] == VER_TAREFA and c["seed"] == VER_SEED][0]
r = aval[(aval.modelo == VER_MODELO.split("/")[-1]) & (aval.tarefa == VER_TAREFA) & (aval.seed == VER_SEED)].iloc[0]
print(f"{VER_MODELO} | {VER_TAREFA} | seed {VER_SEED} | resultado: {r.motivo}\n")
for m in c["mensagens"][1:]:
    if m["role"] == "user":
        print("USUÁRIO  :", m["content"])
    elif m["role"] == "tool":
        print("FERRAMENTA:", m["content"][:200])
    elif m.get("tool_calls"):
        for tc in m["tool_calls"]:
            print("MODELO   -> chama", tc["function"]["name"], json.dumps(tc["function"]["arguments"], ensure_ascii=False))
    else:
        print("MODELO   :", m["content"][:300])
print("\nE-mails enviados:", c["estado"]["emails"])
print("Agenda final:", [(ev["id"], ev["titulo"], ev["data"], ev["hora"]) for ev in c["estado"]["agenda"]])

## 9. Ponte para a Fase 2: os mesmos passos em casos do BFCL

Os casos `bfcl_*` passaram pela mesma geração, pelo mesmo parser e pelo mesmo checker. Isto **não** é o número oficial do BFCL: é uma amostra pequena, com o nosso parser e a nossa normalização. Serve como primeira evidência de execução e como teste do nosso checker contra o gabarito oficial.

Próximo passo da reprodução: rodar o avaliador oficial (`bfcl-eval`) nas mesmas saídas e medir **concordância** entre os dois checkers, caso a caso. Onde discordarem, está a diferença de método.

In [ ]:
m_bfcl = metricas[(metricas.fonte == "bfcl") & (metricas.modo == "estrito")]
if len(m_bfcl):
    display(m_bfcl.pivot(index="categoria", columns="modelo", values="acuracia")[MODELOS].round(2))
    comp_fontes = (df.assign(ok=df.erro_estrito == "ok").groupby(["fonte", "modelo"]).ok.mean().unstack()[MODELOS].round(3))
    print("Acerto geral por fonte (atenção: casos diferentes, categorias em proporções diferentes, idiomas diferentes)")
    display(comp_fontes)
    print("Erros mais comuns nos casos do BFCL:")
    display(df[(df.fonte == "bfcl") & (df.erro_estrito != "ok")].groupby(["modelo", "erro_estrito"]).size().unstack(fill_value=0))
else:
    print("Casos do BFCL não foram carregados nesta execução.")

## 10. Registro de execuções e relatório final

In [ ]:
registro = []
for f in sorted(glob.glob(f"{OUT}/*.json")):
    r = json.load(open(f))
    if "config_hash" in r and "tipo" in r:
        registro.append(dict(arquivo=os.path.basename(f), tipo=r["tipo"], modelo=r["modelo"], config=r["config_hash"],
                             dataset=r.get("dataset_hash", DATASET_HASH), segundos=round(r.get("segundos", 0)), timestamp=r["timestamp"][:19]))
registro = pd.DataFrame(registro)
registro.to_csv(f"{OUT}/registro_execucoes.csv", index=False)
display(registro)

resumo = dict(
    config_hash=CONFIG_HASH, dataset_hash=DATASET_HASH, bfcl_hashes=BFCL_HASHES, env=ENV, modelos=MODELOS,
    single_por_categoria=m_pt[["modelo", "categoria", "n", "acertos", "acuracia", "ic_lo", "ic_hi", "selecao", "args_dado_selecao"]].round(4).to_dict("records"),
    single_capacidades=resumo_single.round(4).to_dict("records"),
    single_parser=comp.round(4).reset_index().to_dict("records"),
    single_erros=cont.reset_index().to_dict("records"),
    exemplos_manuais=dida[["id", "rotulo", "erro_estrito", "erro_tolerante", "schema"]].to_dict("records"),
    multi_sucesso_por_tarefa=aval.groupby(["modelo", "tarefa"]).sucesso.mean().round(3).reset_index().to_dict("records"),
    multi_motivos=aval.groupby(["modelo", "motivo"]).size().reset_index(name="n").to_dict("records"),
    multi_passk=passk.round(4).to_dict("records"),
    bfcl_por_categoria=m_bfcl[["modelo", "categoria", "n", "acertos", "acuracia"]].round(4).to_dict("records"),
)
json.dump(resumo, open(f"{OUT}/resumo_semana8.json", "w"), ensure_ascii=False, indent=1, default=str)

print("RELATÓRIO (casos em português, parser estrito)")
for r in resumo_single.itertuples():
    pk = passk[passk.modelo == r.modelo].set_index("k").pass_k
    print(f"- {r.modelo}: seleção {r.selecao:.2f} | argumentos dado seleção {r.argumentos_dado_selecao:.2f} | "
          f"abstenção correta {r.abstencao_correta:.2f} | multi-turn pass^1 {pk.loc[1]:.2f} -> pass^{n_seeds} {pk.loc[n_seeds]:.2f}")
print(f"- Parser: acerto geral muda de {comp.ok_estrito.round(2).tolist()} (estrito) para {comp.ok_tolerante.round(2).tolist()} (tolerante).")
print(f"- Limitações: 10 casos por categoria (IC largo), casos escritos pela equipe, checagem de resposta por palavras-chave, "
      f"ambiente simulado, um idioma principal, {len(MODELOS)} modelos da mesma família.")
print(f"- Reprodução: config_hash={CONFIG_HASH}, dataset_hash={DATASET_HASH}; saídas brutas e manifests em {OUT}; resumo em resumo_semana8.json.")